In [ ]:
from pathlib import Path
import sys
import warnings

# Resolve the project root whether the notebook is run from /notebooks or the repo root.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Cannot locate project root containing the src/ directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "Student_Performance.csv"
MODELS_DIR = PROJECT_ROOT / "models"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import json
import math
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import f_oneway
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import f_regression, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")


# Notebook Goal
This notebook documents the end-to-end modeling workflow for the Student Performance Predictor:

- inspect and clean the tabular dataset
- split train/test before EDA to reduce leakage risk
- use only the training set for EDA and feature-selection diagnostics
- build the preprocessing pipeline and model comparison workflow
- select a deployable final pipeline using scikit-learn/joblib

Phase 1 cleanup keeps the main modeling direction unchanged. Artifact export is disabled by default to avoid overwriting model files while cleaning the notebook.


# 1. Data Loading


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


# 2. Data Cleaning
Cleaning in this phase is intentionally conservative: remove exact duplicates and enforce the upper bound for the target score. Other outliers are not removed automatically without a clear domain rule.


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
top1_students = df[df['Exam_Score'].between(90, 100)]
print(f"Number of students scoring 90-100: {len(top1_students)}")
top2_students = df[df['Exam_Score'].between(80,89)]
print(f"Number of students scoring 80-89: {len(top2_students)}")
top3_students = df[df['Exam_Score'].between(70,79)]
print(f"Number of students scoring 70-79: {len(top3_students)}")
top4_students = df[df['Exam_Score'] < 70]
print(f"Number of students scoring below 70: {len(top4_students)}")


In [ ]:
df = df.drop_duplicates()
df = df[df['Exam_Score'] <= 100]
df.isnull().sum()


# 3. Train/Test Split
The split is performed before EDA, feature selection, and model comparison. This keeps the test set as an independent holdout and prevents exploratory decisions from learning from evaluation data.


In [ ]:
# Keep the raw split for the deployable sklearn Pipeline.
X = df.drop(columns="Exam_Score")
y = df["Exam_Score"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Analysis copies are used for EDA and feature-selection diagnostics only.
# The final deployable pipeline uses the same raw feature contract.
X_train_analysis = X_train_raw.copy()
X_test_analysis = X_test_raw.copy()

print(f"Raw train shape: {X_train_raw.shape}")
print(f"Analysis train shape: {X_train_analysis.shape}")
print(f"Duplicated rows in analysis train: {X_train_analysis.duplicated().sum()}")


In [ ]:
full_num_features = X_train_analysis.select_dtypes(include=["number"]).columns.tolist()

bin_features_master = [
    "Extracurricular_Activities",
    "Internet_Access",
    "Learning_Disabilities",
    "Gender",
    "School_Type",
]

ord_features_master = [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality",
    "Peer_Influence",
    "Parental_Education_Level",
    "Distance_from_Home",
]

ord_categories_master = [["missing", "Low", "Medium", "High"]] * 5 + [
    ["missing", "Negative", "Neutral", "Positive"],
    ["missing", "High School", "College", "Postgraduate"],
    ["missing", "Near", "Moderate", "Far"],
]

print(
    "Feature groups initialized: "
    f"{len(full_num_features)} numeric, "
    f"{len(ord_features_master)} ordinal, "
    f"{len(bin_features_master)} binary/nominal."
)


## Numeric and Categorical Feature Groups


In [ ]:
num_features = X_train_analysis.select_dtypes(include=np.number).columns
cat_features = X_train_analysis.select_dtypes(exclude=np.number).columns

print("Numeric feature list: ", num_features)
print("Categorical feature list: ", cat_features)


# 4. Exploratory Data Analysis
EDA uses only the training split. The goal is to identify distribution issues, target relationships, encoding needs, and feature-selection signals, not to create unnecessary charts.


## Target and Numeric Distributions
These histograms show the target and numeric predictor distributions. They support scaling decisions and help identify extreme values that need review. This is descriptive EDA; final preprocessing is still fitted through sklearn transformers.


In [ ]:
# 1. Plot the target variable first
plt.figure(figsize=(8, 4))
sns.histplot(y_train, kde=True, color='salmon')
plt.title("Distribution of target variable (Exam_Score)")
plt.show()

# 2. Iterate through the full numeric feature list so no column is missed
for i in num_features:
    # Check whether column i is actually in the DataFrame to avoid errors
    if i in X_train_analysis.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(X_train_analysis[i], kde=True, color='skyblue')
        plt.title(f"Distribution of {i}")
        plt.show()
    else:
        print(f"Warning: Could not find column {i} in X_train_analysis")


## Categorical Feature Frequencies
Count plots help check category imbalance and rare categories before encoding. If rare or missing categories appear, one-hot columns may be less stable on the holdout split. This insight supports the use of imputers and handle_unknown in the encoder.


In [ ]:
n = len(cat_features)
n_cols = 3
n_rows = math.ceil(n / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    # Draw the chart
    sns.countplot(data=X_train_analysis, x=col, ax=axes[i], palette='coolwarm', hue=col)
    
    # Hide redundant legends in each subplot
    if axes[i].get_legend():
        axes[i].get_legend().remove()
        
    axes[i].set_title(f"Frequency: {col}")
    axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=45)

# Remove empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Numeric Outlier Scan
Boxplots are used for a quick numeric outlier scan. This phase does not automatically remove outliers because there is no rule proving they are invalid. Median imputation and a regularized linear model are safer baseline choices.


In [ ]:
# Use num_features, which excludes Exam_Score
n = len(num_features)
n_cols = 3
n_rows = math.ceil(n / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_features):
    # Draw a horizontal boxplot
    sns.boxplot(data=X_train_analysis, x=col, ax=axes[i], color='lightblue', fliersize=5)
    axes[i].set_title(f'Outliers: {col}')
    axes[i].set_xlabel("")

# Remove unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Numeric Relationships with Target
LOWESS/regression plots check whether numeric features have roughly linear relationships with Exam_Score. This supports the Ridge baseline and shows when nonlinear models may be worth trying. These plots are exploratory and are not used to tune on the test set.


In [ ]:
plot_df = pd.concat([X_train_analysis, y_train], axis=1)
n_cols = 3
n_rows = math.ceil(len(num_features) / n_cols)

plt.figure(figsize=(5 * n_cols, 4 * n_rows))

for i, col in enumerate(num_features, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.regplot(
        x=col,
        y="Exam_Score",
        data=plot_df,
        lowess=True,
        scatter_kws={"alpha": 0.2, "s": 10},
        line_kws={"color": "red", "lw": 2},
    )
    plt.title(f"Relationship: {col}")
    plt.xlabel(col)
    plt.ylabel("Exam_Score")

plt.suptitle("Numeric Feature Relationships with Exam Score", fontsize=16, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Numeric Correlation
The correlation heatmap clarifies linear relationships between numeric features and the target. It is useful for detecting redundant predictors before using a linear model. Correlation does not replace feature selection because it may miss nonlinear or categorical effects.


In [ ]:
# Set a readable figure size, for example 10 inches x 6 inches
temp_df = pd.concat([X_train_analysis, y_train], axis=1)

plt.figure(figsize=(10, 6), dpi=100) # 1000x600 pixels
corr = temp_df.corr(numeric_only=True)
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation matrix between numeric variables')
plt.show()


## Mutual Information
Mutual Information is a nonlinear signal for feature relevance and is fitted only on the training set. Temporary ordinal encoding here is used only for diagnostics, not as the final preprocessing representation. MI can be noisy, so it is combined with p-values and correlation filtering.


In [ ]:
# =================================================================
# SECTION: MUTUAL INFORMATION FOR NONLINEAR FEATURE SCREENING

# 1. Create a training-data copy for diagnostics to avoid data leakage
# Do not mutate X_train directly so the pipeline input remains clean
X_mi = X_train_analysis.copy()
y_mi = y_train.copy()

# 2. Handle missing values for categorical variables
# MI does not accept NaN values, so fill them temporarily with the string 'Missing'
X_mi[cat_features] = X_mi[cat_features].fillna('Missing')

# 3. Temporarily encode all text variables
# For MI diagnostics, OrdinalEncoder is acceptable for both nominal and binary variables.
# MI only needs to distinguish groups and does not rely on order or distance.
temp_label_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_mi[cat_features] = temp_label_encoder.fit_transform(X_mi[cat_features].astype(str))

# 4. Create a discrete mask so MI knows which columns are categorical
# This is important because categorical features follow a discrete distribution
discrete_mask = [col in cat_features for col in X_mi.columns]

# 5. Calculate Mutual Information scores
# random_state=42 keeps the result reproducible
mi_scores = mutual_info_regression(
    X_mi, y_mi, 
    discrete_features=discrete_mask, 
    random_state=42
)

# 6. Put the result into a DataFrame and sort descending
mi_df = pd.DataFrame({
    'Feature': X_mi.columns, 
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False)

# 7. Visualize the result with a bar plot
plt.figure(figsize=(10, 8))

# Draw the chart and keep the axis object for annotation
ax = sns.barplot(data=mi_df, x='MI_Score', y='Feature', palette='viridis')

# Add numbers at the end of each bar
# padding=3 moves the text slightly away from the bar
# fmt='%.4f' keeps four decimal places because MI scores are usually small
ax.bar_label(ax.containers[0], padding=3, fmt='%.4f', fontsize=10)

# Add chart styling
plt.title('Mutual Information contribution to Exam_Score', fontsize=14, pad=20)
plt.xlabel('MI Score', fontsize=12)
plt.ylabel('Features', fontsize=12)

# Expand the x-axis slightly so annotations do not touch the chart edge
plt.xlim(0, mi_df['MI_Score'].max() * 1.15) 

plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## ANOVA for Categorical Features
ANOVA checks whether the target mean differs across category groups. The result helps identify categorical predictors worth keeping after encoding. The p-value is a screening signal, not causal evidence.


In [ ]:
# =================================================================
# SECTION: ANOVA TEST FOR CATEGORICAL FEATURES
# =================================================================

# 1. Prepare data from the training split to avoid leakage
temp_df = pd.concat([X_train_analysis, y_train], axis=1)
target_col = 'Exam_Score'
anova_list = []

print("Running ANOVA tests...")

for col in cat_features:
    # 2. Group values and filter out tiny samples to avoid SmallSampleWarning
    # Each group needs at least 2 samples to estimate within-group variance
    valid_groups = []
    for val in temp_df[col].unique():
        group = temp_df[temp_df[col] == val][target_col].dropna()
        if len(group) > 1:
            valid_groups.append(group)
    
    # 3. Run f_oneway only if at least 2 groups can be compared
    if len(valid_groups) >= 2:
        try:
            stat, p = f_oneway(*valid_groups)
            
            # Handle extremely small p-values to avoid math issues when computing Log10
            # 1e-15 is small enough to represent very strong differences
            p_safe = max(p, 1e-15) if not np.isnan(p) else np.nan
            
            anova_list.append({
                'Feature': col, 
                'p_value': p, 
                'Significance_Score': -np.log10(p_safe)
            })
        except Exception:
            continue

# 4. Put results into a DataFrame and sort by importance
anova_df = pd.DataFrame(anova_list).dropna()
anova_df = anova_df.sort_values('Significance_Score', ascending=False)

# 5. Visualize the result
plt.figure(figsize=(12, 10)) 
ax = sns.barplot(
    data=anova_df, 
    x='Significance_Score', 
    y='Feature', 
    palette='magma'
)

# 6. Write score and p-value directly on the chart
for i, (score, p) in enumerate(zip(anova_df['Significance_Score'], anova_df['p_value'])):
    text_color = 'darkgreen' if p <= 0.05 else 'darkred'
    # Show both score and p-value in scientific notation
    display_text = f'Score: {score:.2f} | p: {p:.2e}'
    
    ax.text(
        score + 0.1, i, 
        display_text, 
        va='center', 
        fontsize=10, 
        fontweight='bold',
        color=text_color
    )

# 7. CHART STYLING AND DISPLAY
# Threshold line for p = 0.05
plt.axvline(x=-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='Significance threshold (p=0.05)')

plt.title('ANOVA Analysis: Categorical Feature Signal', fontsize=16, pad=25)
plt.xlabel('Significance Score [-log10(p-value)]', fontsize=12)
plt.ylabel('Categorical Features', fontsize=12)

# Expand the x-axis to prevent text clipping
plt.xlim(0, anova_df['Significance_Score'].max() * 1.5)

plt.legend(loc='lower right')
plt.grid(axis='x', linestyle='--', alpha=0.3)

# Final display command
plt.tight_layout()
plt.show()

print("ANOVA analysis completed.")


## Ordinal Monotonicity Check
These plots check whether ordinally encoded categories follow a reasonable trend with the target. If the order is not monotonic, ordinal encoding may introduce a wrong assumption. This is evidence for reviewing preprocessing, not a hard guarantee.


In [ ]:
# STEP 4: ORDINAL MONOTONICITY CHECK
ordinal_mapping = {
    'Parental_Involvement': ['Low', 'Medium', 'High'],
    'Access_to_Resources': ['Low', 'Medium', 'High'],
    'Motivation_Level': ['Low', 'Medium', 'High'],
    'Family_Income': ['Low', 'Medium', 'High'],
    'Teacher_Quality': ['Low', 'Medium', 'High'],
    'Peer_Influence': ['Negative', 'Neutral', 'Positive'],
    'Parental_Education_Level': ['High School', 'College', 'Postgraduate'],
    'Distance_from_Home': ['Near', 'Moderate', 'Far']
}
# =================================================================
print("Checking ordinal monotonicity...")
for col, order in ordinal_mapping.items():
    if col in temp_df.columns:
        plt.figure(figsize=(8, 5))
        # Combine boxplot and pointplot to show the trend line
        sns.boxplot(data=temp_df, x=col, y=target_col, order=order, palette='Pastel1', showfliers=False)
        sns.pointplot(data=temp_df, x=col, y=target_col, order=order, color='red', markers='D', linestyles='--')
        
        plt.title(f'3. Monotonicity Check: {col}', fontsize=12)
        plt.grid(axis='y', alpha=0.3)
        plt.show()


## Interaction Checks
A small set of interaction plots is kept to check whether strong predictors behave differently across categorical groups. The notebook avoids expanding too many charts to prevent EDA bloat. Phase 1 does not add new interaction features.


In [ ]:
# 1. Prepare data
temp_df = pd.concat([X_train_analysis, y_train], axis=1)

def plot_interaction_pro(df, feat1, feat2, target, order1=None, hue_order=None):
    plt.figure(figsize=(12, 7))
    df_plot = df.copy()
    
    # --- HANDLE X-AXIS FEATURE (FEAT1) ---
    if pd.api.types.is_numeric_dtype(df_plot[feat1]):
        # Bin only numeric features
        df_plot[feat1] = pd.qcut(df_plot[feat1], q=4, duplicates='drop')
        x_order = sorted(df_plot[feat1].unique())
        xlabel = f"{feat1} (Grouped by quartile)"
    else:
        # If the feature is categorical, keep its original values for plotting
        x_order = None # Or define a specific order, such as ['Low', 'Medium', 'High']
        xlabel = feat1

    # --- DRAW CHART ---
    sns.pointplot(
        data=df_plot, 
        x=feat1, 
        y=target, 
        hue=feat2,
        order=x_order, 
        hue_order=hue_order,
        palette='magma', 
        capsize=.1, 
        markers=["o", "s", "D"],
        linestyles=["-", "--", "-."]
    )

    # --- CHART STYLING ---
    plt.title(f'Interaction Effect Analysis: {feat1} & {feat2}', fontsize=15, pad=20)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(f'Average {target}', fontsize=12)
    plt.legend(title=feat2, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

# =================================================================
# RUN CHECKS WITH THE RIGHT PARAMETERS
# =================================================================

# Pair 1: Two strong categorical variables
print("Analyzing: Parental_Involvement & Access_to_Resources")
plot_interaction_pro(
    temp_df, 
    feat1='Parental_Involvement', 
    feat2='Access_to_Resources', 
    target='Exam_Score',
    order1=['Low', 'Medium', 'High'],
    hue_order=['Low', 'Medium', 'High']
)

# Pair 2: Strong numeric variable and categorical variable
print("Analyzing: Hours_Studied & Motivation_Level")
plot_interaction_pro(
    temp_df, 
    feat1='Hours_Studied', 
    feat2='Motivation_Level', 
    target='Exam_Score',
    hue_order=['Low', 'Medium', 'High']
)

# Pair 3: Strong numeric variable and resource access
print("Analyzing: Attendance & Access_to_Resources")
plot_interaction_pro(
    temp_df, 
    feat1='Attendance', 
    feat2='Access_to_Resources', 
    target='Exam_Score',
    hue_order=['Low', 'Medium', 'High']
)


# 5. Preprocessing and Feature Selection
Phase 1 keeps the existing hybrid feature-selection approach. Diagnostics are fitted on the train split and then mapped back to raw feature names so the final pipeline still accepts the same raw-input contract as the app.


In [ ]:
current_num = [c for c in full_num_features if c in X_train_analysis.columns]
current_bin = [c for c in bin_features_master if c in X_train_analysis.columns]

current_ord = []
current_ord_categories = []
for i, col in enumerate(ord_features_master):
    if col in X_train_analysis.columns:
        current_ord.append(col)
        current_ord_categories.append(ord_categories_master[i])

if len(current_num) + len(current_bin) + len(current_ord) == 0:
    raise ValueError("No feature groups matched the training data.")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            current_num,
        ),
        (
            "ord",
            Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                ("enc", OrdinalEncoder(categories=current_ord_categories)),
            ]),
            current_ord,
        ),
        (
            "bin",
            Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                ("enc", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
            ]),
            current_bin,
        ),
    ],
    remainder="drop",
)

print(
    "Preprocessor groups: "
    f"{len(current_num)} numeric, {len(current_ord)} ordinal, {len(current_bin)} binary/nominal."
)


In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train_analysis)
X_test_encoded = preprocessor.transform(X_test_analysis)

_, p_values = f_regression(X_train_encoded, y_train)
mi_scores = mutual_info_regression(X_train_encoded, y_train, random_state=42)

selection_df = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "p_value": p_values,
    "MI_Score": mi_scores,
})


def clean_feature_names(name):
    clean_name = name.split("__")[-1]
    suffixes = [
        "_Yes",
        "_No",
        "_Male",
        "_Female",
        "_Public",
        "_Private",
        "_Urban",
        "_Rural",
    ]
    for suffix in suffixes:
        if clean_name.endswith(suffix):
            clean_name = clean_name.replace(suffix, "")
    return clean_name


selection_df["Clean_Feature"] = selection_df["Feature"].apply(clean_feature_names)
selection_df = selection_df.sort_values(by="MI_Score", ascending=False).reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(14, 7))
ax2 = ax1.twinx()

ax1.bar(
    selection_df["Clean_Feature"],
    selection_df["MI_Score"],
    color="#5cb85c",
    alpha=0.7,
    label="Mutual Information",
)
ax2.plot(
    selection_df["Clean_Feature"],
    selection_df["p_value"],
    color="#d9534f",
    marker="o",
    linewidth=2,
    label="p-value",
)

ax1.axhline(y=0.01, color="green", linestyle="--", alpha=0.5)
ax2.axhline(y=0.05, color="red", linestyle="--", alpha=0.5)
ax1.tick_params(axis="x", labelrotation=45)

ax1.set_ylabel("MI Score", fontsize=12, fontweight="bold", color="green")
ax2.set_ylabel("p-value", fontsize=12, fontweight="bold", color="red")
plt.title("Hybrid Feature Screening: F-test vs Mutual Information", fontsize=15, pad=20)

lns1, labs1 = ax1.get_legend_handles_labels()
lns2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lns1 + lns2, labs1 + labs2, loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

selected_mask = (selection_df["p_value"] < 0.05) | (selection_df["MI_Score"] > 0.01)
selected_df = selection_df[selected_mask].copy()
encoded_selected_features = selected_df["Feature"].tolist()

X_train_selected = X_train_encoded[encoded_selected_features].copy()
X_test_selected = X_test_encoded[encoded_selected_features].copy()

corr_matrix = X_train_selected.corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

corr_threshold = 0.85
features_to_drop = []

for col in upper_triangle.columns:
    high_corr_features = upper_triangle.index[upper_triangle[col] > corr_threshold].tolist()
    for row_feature in high_corr_features:
        row_info = selected_df[selected_df["Feature"] == row_feature]
        col_info = selected_df[selected_df["Feature"] == col]
        if row_info.empty or col_info.empty:
            continue

        row_mi = row_info["MI_Score"].values[0]
        col_mi = col_info["MI_Score"].values[0]
        row_p = row_info["p_value"].values[0]
        col_p = col_info["p_value"].values[0]

        if row_mi > col_mi:
            drop_feature = col
        elif row_mi < col_mi:
            drop_feature = row_feature
        else:
            drop_feature = col if row_p > col_p else row_feature

        if drop_feature not in features_to_drop:
            features_to_drop.append(drop_feature)

X_train_clean = X_train_selected.drop(columns=features_to_drop, errors="ignore")
X_test_clean = X_test_selected.drop(columns=features_to_drop, errors="ignore")

display_feature_names = [clean_feature_names(col) for col in X_train_clean.columns]

print(f"Selected encoded features before correlation filter: {len(encoded_selected_features)}")
print(f"Dropped highly correlated features: {len(features_to_drop)}")
print(f"Final selected encoded features: {X_train_clean.shape[1]}")
print(display_feature_names)


In [ ]:
def encoded_to_raw_feature(feature_name):
    raw_name = feature_name.split("__")[-1]
    known_suffixes = [
        "_Yes",
        "_No",
        "_Male",
        "_Female",
        "_Public",
        "_Private",
        "_Urban",
        "_Rural",
    ]
    for suffix in known_suffixes:
        if raw_name.endswith(suffix):
            raw_name = raw_name.replace(suffix, "")
    return raw_name


encoded_survivors = X_train_clean.columns.tolist()
raw_survivors = [encoded_to_raw_feature(col) for col in encoded_survivors]
raw_survivors = list(dict.fromkeys(raw_survivors))

current_num = [c for c in full_num_features if c in raw_survivors]
current_bin = [c for c in bin_features_master if c in raw_survivors]

current_ord = []
current_ord_categories = []
for i, col in enumerate(ord_features_master):
    if col in raw_survivors:
        current_ord.append(col)
        current_ord_categories.append(ord_categories_master[i])

if len(current_num) + len(current_ord) + len(current_bin) == 0:
    raise ValueError("No raw features left after mapping encoded survivors to raw features.")

preprocessor_selected = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            current_num,
        ),
        (
            "ord",
            Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                ("enc", OrdinalEncoder(categories=current_ord_categories)),
            ]),
            current_ord,
        ),
        (
            "bin",
            Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                ("enc", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
            ]),
            current_bin,
        ),
    ],
    remainder="drop",
)

print(f"Raw survivors: {raw_survivors}")
print(
    "Selected preprocessor groups: "
    f"{len(current_num)} numeric, {len(current_ord)} ordinal, {len(current_bin)} binary/nominal."
)


In [ ]:
X_train_selected_encoded = preprocessor_selected.fit_transform(X_train_analysis[raw_survivors])
X_test_selected_encoded = preprocessor_selected.transform(X_test_analysis[raw_survivors])

print("X_train_selected_encoded shape:", X_train_selected_encoded.shape)
print("X_test_selected_encoded shape :", X_test_selected_encoded.shape)

print(X_train_selected_encoded.head())


# 6. Multicollinearity Check
VIF is a sanity check for selected encoded features before using linear models. A high VIF does not automatically require removing a feature, but it helps explain why Ridge is more suitable than plain Linear Regression.


In [ ]:
def check_vif_optimized(X_df):
    X = X_df.select_dtypes(include=[np.number]).copy()
    constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
    X = X.drop(columns=constant_cols, errors="ignore")

    if X.shape[1] == 0:
        raise ValueError("No numeric columns left for VIF calculation.")

    X_with_const = add_constant(X)
    vif_data = pd.DataFrame({
        "Feature": X_with_const.columns,
        "VIF": [
            variance_inflation_factor(X_with_const.values, i)
            for i in range(X_with_const.shape[1])
        ],
    })

    vif_results = vif_data[vif_data["Feature"] != "const"].copy()
    vif_results["Display Name"] = vif_results["Feature"].apply(clean_feature_names)

    def get_status(vif):
        if vif < 5:
            return "Low"
        if vif < 10:
            return "Moderate"
        return "High"

    vif_results["Status"] = vif_results["VIF"].apply(get_status)
    return vif_results[["Feature", "Display Name", "VIF", "Status"]].sort_values(
        by="VIF",
        ascending=False,
    )


vif_report = check_vif_optimized(X_train_selected_encoded)
display(vif_report)

high_vif_count = (vif_report["VIF"] > 5).sum()
print(f"Features with VIF > 5: {high_vif_count}")


In [ ]:
train_data = pd.concat([X_train_raw.copy(), y_train.copy()], axis=1)

score_band_counts = pd.Series({
    "90+": (train_data["Exam_Score"] >= 90).sum(),
    "80-89": ((train_data["Exam_Score"] >= 80) & (train_data["Exam_Score"] < 90)).sum(),
    "70-79": ((train_data["Exam_Score"] >= 70) & (train_data["Exam_Score"] < 80)).sum(),
    "<70": (train_data["Exam_Score"] < 70).sum(),
})

display(score_band_counts.to_frame("Train Count"))

X_train_for_model = X_train_raw.copy()
y_train_for_model = y_train.copy()

print(f"Modeling train shape: {X_train_for_model.shape}")


In [ ]:
X_train_for_model_eng = X_train_for_model[raw_survivors].copy()
X_test_for_model_eng = X_test_raw[raw_survivors].copy()

X_train_model_encoded = preprocessor_selected.fit_transform(X_train_for_model_eng)
X_test_model_encoded = preprocessor_selected.transform(X_test_for_model_eng)

print("Encoded train shape:", X_train_model_encoded.shape)
print("Encoded test shape:", X_test_model_encoded.shape)


# 7. Model Selection
The notebook compares a linear baseline, regularized linear models, and tree-based alternatives. Ridge Regression is preferred for the final portfolio model when performance is competitive because it is lightweight, explainable, more stable under multicollinearity, and easy to deploy in an sklearn Pipeline.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd

models = {}
cv_scores = {}

# =========================
# 1. Dummy Baseline
# =========================
print("[1/6] Training Dummy Regressor baseline...")
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train_model_encoded, y_train_for_model)
models["Dummy Baseline"] = dummy_model

dummy_cv_scores = cross_val_score(
    dummy_model,
    X_train_model_encoded,
    y_train_for_model,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
cv_scores["Dummy Baseline"] = dummy_cv_scores.mean()


# =========================
# 2. Linear Regression
# =========================
print("[2/6] Training Linear Regression...")
linear_model = LinearRegression()
linear_model.fit(X_train_model_encoded, y_train_for_model)
models["Linear Regression"] = linear_model

linear_cv_scores = cross_val_score(
    linear_model,
    X_train_model_encoded,
    y_train_for_model,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
cv_scores["Linear Regression"] = linear_cv_scores.mean()


# =========================
# 3. Lasso Regression
# =========================
print("[3/6] Tuning Lasso Regression...")
lasso_grid = GridSearchCV(
    estimator=Lasso(random_state=42, max_iter=10000),
    param_grid={
        "alpha": [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
)

lasso_grid.fit(X_train_model_encoded, y_train_for_model)

models["Lasso"] = lasso_grid.best_estimator_
cv_scores["Lasso"] = lasso_grid.best_score_


# =========================
# 4. Ridge Regression
# =========================
print("[4/6] Tuning Ridge Regression...")
ridge_grid = GridSearchCV(
    estimator=Ridge(random_state=42),
    param_grid={
        "alpha": [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
)

ridge_grid.fit(X_train_model_encoded, y_train_for_model)

models["Ridge"] = ridge_grid.best_estimator_
cv_scores["Ridge"] = ridge_grid.best_score_


# =========================
# 5. Random Forest
# =========================
print("[5/6] Tuning Random Forest...")
rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid={
        "n_estimators": [200, 500],
        "max_depth": [None, 5, 8, 12],
        "min_samples_leaf": [2, 5, 10],
        "max_features": ["sqrt", 0.7, 1.0],
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
)

rf_grid.fit(X_train_model_encoded, y_train_for_model)

models["Random Forest"] = rf_grid.best_estimator_
cv_scores["Random Forest"] = rf_grid.best_score_


# =========================
# 6. XGBoost Optional
# =========================
try:
    from xgboost import XGBRegressor

    print("[6/6] Tuning XGBoost...")

    xgb_grid = GridSearchCV(
        estimator=XGBRegressor(
            random_state=42,
            tree_method="hist",
            objective="reg:squarederror",
        ),
        param_grid={
            "n_estimators": [300, 500],
            "learning_rate": [0.01, 0.03],
            "max_depth": [2, 3],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "reg_lambda": [10, 30, 50],
            "min_child_weight": [5, 10],
        },
        cv=5,
        scoring="r2",
        n_jobs=-1,
    )

    xgb_grid.fit(X_train_model_encoded, y_train_for_model)

    models["XGBoost"] = xgb_grid.best_estimator_
    cv_scores["XGBoost"] = xgb_grid.best_score_

except ImportError:
    print("[6/6] XGBoost is not installed. Skipping XGBoost.")
    xgb_grid = None


# =========================
# Model Evaluation
# =========================
results = []

for model_name, model in models.items():
    y_pred_train = model.predict(X_train_model_encoded)
    y_pred_test = model.predict(X_test_model_encoded)

    r2_train = r2_score(y_train_for_model, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = mean_squared_error(y_test, y_pred_test) ** 0.5

    results.append({
        "Model": model_name,
        "CV R2": cv_scores.get(model_name),
        "R2 Train": r2_train,
        "R2 Test": r2_test,
        "Gap (Train-Test)": r2_train - r2_test,
        "MAE": mae,
        "RMSE": rmse,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(by="R2 Test", ascending=False)
    .reset_index(drop=True)
)

display(results_df)


# =========================
# Best Hyperparameters Summary
# =========================
print("\nBest hyperparameters:")

print("Best Lasso params:", lasso_grid.best_params_)
print("Best Lasso CV R2:", lasso_grid.best_score_)

print("Best Ridge params:", ridge_grid.best_params_)
print("Best Ridge CV R2:", ridge_grid.best_score_)

print("Best Random Forest params:", rf_grid.best_params_)
print("Best Random Forest CV R2:", rf_grid.best_score_)

if xgb_grid is not None:
    print("Best XGBoost params:", xgb_grid.best_params_)
    print("Best XGBoost CV R2:", xgb_grid.best_score_)

In [ ]:
# 1. Get the best Ridge model
best_ridge = ridge_grid.best_estimator_

# 2. Get feature names from the final encoded matrix
encoded_feature_names = X_train_model_encoded.columns.tolist()

# 3. Clean feature names for display
def clean_display_name(name):
    clean_name = name.split("__")[-1]

    suffixes = [
        "_Yes", "_No",
        "_Male", "_Female",
        "_Public", "_Private",
        "_Urban", "_Rural"
    ]

    for s in suffixes:
        if clean_name.endswith(s):
            clean_name = clean_name.replace(s, "")

    return clean_name

display_names = [clean_display_name(name) for name in encoded_feature_names]

# 4. Build a Series from Ridge coefficients
coef_series = pd.Series(best_ridge.coef_, index=display_names)

# Sum coefficients when display names are duplicated
coef_series = coef_series.groupby(level=0).sum()

# Use absolute coefficient size to rank feature influence
coef_abs = coef_series.abs().sort_values(ascending=False)

# 5. Select the top 10 most important features
top_features = coef_abs.nlargest(10).index
top_coef = coef_series.loc[top_features].sort_values()

# 6. Color by effect direction
colors = ["#E74C3C" if val < 0 else "#2ECC71" for val in top_coef]

# 7. Draw the chart
plt.figure(figsize=(10, 6))
top_coef.plot(
    kind="barh",
    color=colors
)

plt.title("Top 10 Most Important Ridge Regression Features", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Effect Coefficient")
plt.ylabel("Feature")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 1. FINAL MODEL: RIDGE
# =========================================================
best_ridge = ridge_grid.best_estimator_

# =========================================================
# 2. PREDICT ON TEST SET
# =========================================================
y_true = y_test.values if hasattr(y_test, "values") else y_test
y_pred = best_ridge.predict(X_test_model_encoded)

# =========================================================
# 3. METRICS
# =========================================================
r2 = r2_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
abs_errors = np.abs(y_true - y_pred)
within_5 = (abs_errors <= 5).mean() * 100

# =========================================================
# 4. PLOT
# =========================================================
plt.figure(figsize=(10, 8))

min_val = min(y_true.min(), y_pred.min()) - 3
max_val = max(y_true.max(), y_pred.max()) + 3

# Perfect prediction line
plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    color="#d62728",
    label="Perfect Prediction"
)

# Tolerance band ±5
tolerance = 5
plt.fill_between(
    [min_val, max_val],
    [min_val - tolerance, max_val - tolerance],
    [min_val + tolerance, max_val + tolerance],
    color="#d62728",
    alpha=0.08,
    label="±5 Score Band"
)

# Scatter
scatter = plt.scatter(
    y_true,
    y_pred,
    c=abs_errors,
    cmap="viridis",
    s=70,
    alpha=0.85,
    edgecolors="white",
    linewidth=0.6
)

# Colorbar
cbar = plt.colorbar(scatter, pad=0.02)
cbar.set_label("Absolute Error", rotation=270, labelpad=18, fontweight="bold")

# Metrics box
metrics_text = (
    f"R²   = {r2:.3f}\n"
    f"MAE  = {mae:.2f}\n"
    f"RMSE = {rmse:.2f}\n"
    f"Within ±5 = {within_5:.1f}%"
)

plt.text(
    0.04, 0.96,
    metrics_text,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        alpha=0.92,
        edgecolor="#bbbbbb"
    )
)

# Styling
plt.title("Ridge Regression: Actual vs Predicted Student Scores", fontsize=17, fontweight="bold", pad=18)
plt.xlabel("Actual Score", fontsize=12, fontweight="bold")
plt.ylabel("Predicted Score", fontsize=12, fontweight="bold")

plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)

plt.grid(True, linestyle="--", alpha=0.35)
plt.legend(loc="lower right", frameon=True, shadow=True, fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
best_ridge = ridge_grid.best_estimator_

final_ridge = Ridge(alpha=ridge_grid.best_params_["alpha"], random_state=42)

full_pipeline_selected = Pipeline([
    ("preprocess", preprocessor_selected),
    ("model", final_ridge),
])

full_pipeline_selected.fit(X_train_for_model, y_train_for_model)
y_pred_final_pipeline = full_pipeline_selected.predict(X_test_raw)

r2_final_pipeline = r2_score(y_test, y_pred_final_pipeline)
mae_final_pipeline = mean_absolute_error(y_test, y_pred_final_pipeline)
rmse_final_pipeline = mean_squared_error(y_test, y_pred_final_pipeline) ** 0.5

print("Final raw-input pipeline metrics:")
print(f"R2 Score: {r2_final_pipeline:.4f}")
print(f"MAE     : {mae_final_pipeline:.2f}")
print(f"RMSE    : {rmse_final_pipeline:.2f}")


# 8. Limitations
- Feature-selection thresholds are still heuristic and should be validated with cleaner cross-validation in a later phase.
- The holdout test set is used for model comparison, so the final metric should be treated as project evidence, not a locked production benchmark.
- Dataset schema validation and unknown-category handling need additional hardening before production use.
- Ridge is selected because it balances deployability and explainability; this is a portfolio-oriented trade-off, not proof that Ridge is always the most accurate model.


# 9. Artifact Export and Smoke Check
The intended order is: prepare export paths, export the final pipeline, then reload the artifact for a smoke prediction. Phase 1 keeps export disabled by default to avoid overwriting model files during notebook cleanup.


In [ ]:
# Safety flags for notebook review runs.
# Keep both False unless you intentionally want to write or reload artifacts.
EXPORT_ARTIFACTS = False
SMOKE_CHECK_EXPORTED_ARTIFACT = False


In [ ]:
pipeline_path = MODELS_DIR / "hcmue_student_full_pipeline_v1_0.joblib"
export_metadata = {
    "model_name": "Ridge Regression",
    "alpha": ridge_grid.best_params_["alpha"],
    "raw_feature_names": list(X_train_for_model.columns),
    "raw_survivors": raw_survivors,
    "metrics": {
        "r2": float(r2_final_pipeline),
        "mae": float(mae_final_pipeline),
        "rmse": float(rmse_final_pipeline),
    },
}

if EXPORT_ARTIFACTS:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(full_pipeline_selected, pipeline_path)
    joblib.dump(final_ridge, MODELS_DIR / "ridge_core_model.joblib")
    joblib.dump(list(X_train_for_model.columns), MODELS_DIR / "raw_feature_names.joblib")
    joblib.dump(raw_survivors, MODELS_DIR / "raw_survivors.joblib")

    with open(MODELS_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as f:
        json.dump(
            {"model_name": "Ridge Regression", "alpha": ridge_grid.best_params_["alpha"]},
            f,
            indent=4,
            ensure_ascii=False,
        )

    with open(MODELS_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
        json.dump(export_metadata, f, indent=4, ensure_ascii=False)

    print(f"Exported artifacts to: {MODELS_DIR}")
else:
    print("Artifact export skipped. Set EXPORT_ARTIFACTS=True to write files.")


In [ ]:
if SMOKE_CHECK_EXPORTED_ARTIFACT:
    loaded_pipeline = joblib.load(pipeline_path)
    sample_input_ultra = {
        "Hours_Studied": 12,
        "Attendance": 100,
        "Parental_Involvement": "High",
        "Access_to_Resources": "High",
        "Extracurricular_Activities": "Yes",
        "Sleep_Hours": 8,
        "Previous_Scores": 100,
        "Motivation_Level": "High",
        "Internet_Access": "Yes",
        "Tutoring_Sessions": 12,
        "Family_Income": "High",
        "Teacher_Quality": "High",
        "School_Type": "Public",
        "Peer_Influence": "Positive",
        "Physical_Activity": 8,
        "Learning_Disabilities": "No",
        "Parental_Education_Level": "Postgraduate",
        "Distance_from_Home": "Near",
        "Gender": "Male",
    }
    sample_df_ultra = pd.DataFrame([sample_input_ultra], columns=X_train_for_model.columns)
    pred_ultra_loaded = loaded_pipeline.predict(sample_df_ultra)
    print("Prediction from loaded pipeline:", pred_ultra_loaded)
else:
    print("Smoke check skipped. Set SMOKE_CHECK_EXPORTED_ARTIFACT=True after exporting.")
